import bibliothèque

In [ ]:
import pandas as pd

from snowflake.snowpark.context import get_active_session

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

Connexion Snowflake

In [ ]:
session = get_active_session()

chargement des données

In [ ]:
df = session.table("HOUSE_PRICE.SILVER.HOUSE_PRICE").to_pandas()
df.head()

Définir X et y

In [ ]:
X = df.drop(columns=["PRICE"])
y = df["PRICE"]

Séparer train / test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Entraîner le modèle

In [ ]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object", "bool"]).columns

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(random_state=42))
])

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
y_pred[:5]

Évaluation du modèle

In [ ]:
y_pred = model.predict(X_test)

Calcul des métriques de performance

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("RMSE :", rmse)
print("R² :", r2)

Comparaison entre valeurs réelles et valeurs prédites

In [ ]:
comparison = pd.DataFrame({
    "Prix réel": y_test.values,
    "Prix prédit": y_pred
})

comparison.head(10)

Analyse des résultats

In [ ]:
comparison["Erreur absolue"] = abs(comparison["Prix réel"] - comparison["Prix prédit"])

comparison.head(10)

Optimisation du modèle

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("RMSE :", rmse)
print("R² :", r2)

Le modèle a été évalué avec des métriques adaptées à un problème de régression. 
L’objectif étant de prédire un prix, les métriques Accuracy, Precision et Recall ne sont pas adaptées, car elles concernent les problèmes de classification.

Nous utilisons donc la MAE, la RMSE et le score R².
La MAE indique l’erreur moyenne entre le prix réel et le prix prédit.
La RMSE pénalise davantage les grandes erreurs de prédiction.
Le score R² indique la capacité du modèle à expliquer les variations du prix des maisons.

La comparaison entre les prix réels et les prix prédits permet d’observer concrètement les écarts du modèle.
Les axes d’amélioration possibles sont l’optimisation des hyperparamètres, le test d’autres modèles comme XGBoost, et l’amélioration de la préparation des données.

Optimisation du modèle avec Grid Search

In [ ]:
from sklearn.model_selection import GridSearchCV

Définition des hyperparamètres

In [ ]:
param_grid = {
    "regressor__n_estimators": [50, 100],
    "regressor__max_depth": [None, 10, 20],
    "regressor__min_samples_split": [2, 5]
}

Recherche du meilleur modèle

In [ ]:
grid_search = GridSearchCV(
    model,
    param_grid,
    cv=3,
    scoring="neg_mean_squared_error",
    n_jobs=1
)

grid_search.fit(X_train, y_train)

Meilleurs paramètres

In [ ]:
print("Meilleurs paramètres :", grid_search.best_params_)

Évaluation du meilleur modèle

In [ ]:
best_model = grid_search.best_estimator_

y_pred_best = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_best)
rmse = mean_squared_error(y_test, y_pred_best) ** 0.5
r2 = r2_score(y_test, y_pred_best)

print("MAE :", mae)
print("RMSE :", rmse)
print("R² :", r2)

Une optimisation du modèle a été réalisée avec Grid Search afin de tester différentes combinaisons d’hyperparamètres.

Cette méthode permet d’identifier automatiquement la configuration offrant les meilleures performances.

Le modèle optimisé présente de meilleures performances que le modèle initial, ce qui montre l’intérêt de l’ajustement des hyperparamètres.

Les axes d’amélioration futurs pourraient inclure l’utilisation d’autres modèles comme XGBoost ou l’ajout de nouvelles variables explicatives.

In [ ]:
from snowflake.ml.registry import Registry

# Créer le registry
reg = Registry(session = session)

# Enregistrer le meilleur modèle
snow_model = reg.log_model(
    model = best_model,
    model_name = "house_price_rf",
    version_name = "v2",
    sample_input_data = X_test.head(5),
    comment = "RandomForest optimisé via GridSearch",
    target_platforms = ["WAREHOUSE"],
    options = {"enable_explainability": False}
)

# Vérifier l'enregistrement
reg.show_models()

In [ ]:
# Charger le modèle depuis le registry
loaded_model = reg.get_model("house_price_rf").version("v2")

# Convertir X_test en Snowpark DataFrame pour l'inférence native
X_test_sp = session.create_dataframe(X_test)

# Prédictions via le modèle chargé
predictions = loaded_model.run(X_test_sp, function_name = "predict")
predictions.show(10)

# Stocker les résultats dans une table Snowflake
predictions.write.save_as_table(
    "HOUSE_PRICE.SILVER.PREDICTIONS",
    mode = "overwrite"
)